In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from src import load_and_prep_data, HawkesExp, GlobalSeasonalProfile
from src.utils import temporal_train_test_split
from src.metrics import calc_test_ll, evaluate_forecast

In [5]:
all_sequences = load_and_prep_data('data/raw/retail_data.csv')

MIN_EVENTS = 10
active_users_idx = all_sequences[all_sequences.apply(len) >= MIN_EVENTS].index
print(f"Total users: {len(all_sequences)}, Active users (>={MIN_EVENTS}): {len(active_users_idx)}")

In [4]:
print("Fitting Global Seasonality...")

all_active_times = [all_sequences[uid] for uid in active_users_idx]
global_profile = GlobalSeasonalProfile().fit(all_active_times)

Hawkes params: [1.78707623 7.89296502]


In [ ]:
# 4. Цикл валидации
results = []

for uid in tqdm(active_users_idx):
    times = all_sequences[uid]
    
    # Split
    train, test = temporal_train_test_split(times, train_ratio=0.8)
    
    # Если в тесте пусто (такое бывает при редких покупках в конце), пропускаем
    if len(test) < 1: continue

    # --- Baseline: Global Seasonality Scaled for User ---
    # Считаем среднюю частоту юзера на трейне
    duration_hours = (train[-1] - train[0]).total_seconds() / 3600
    if duration_hours < 1: duration_hours = 1
    user_avg_rate = len(train) / duration_hours
    
    # Создаем "адаптер" для baseline, который совместим с интерфейсом HawkesExp
    # (нужен объект с методом get_intensity(t))
    class UserBaseline:
        def __init__(self, glob_prof, rate):
            self.glob = glob_prof
            self.rate = rate
            self.rates = None # Заглушка, если Hawkes захочет mean
        def get_intensity(self, t, history=None):
            return self.glob.get_rate(t, self.rate)
        @property
        def rates(self): # Для интеграла в Hawkes
            return np.array([self.glob.get_rate(pd.Timestamp(2020,1,1,h), self.rate) for h in range(24)])

    baseline_model = UserBaseline(global_profile, user_avg_rate)

    # --- Model 1: Просто Сезонность (Baseline) ---
    # Она ничего не учит, просто использует глобальный профиль
    # Но нам нужен метод nll, поэтому можно обернуть или использовать Hawkes с alpha=0
    # Для простоты сравним Hawkes против Hawkes(alpha=0) -> это и есть Пуассон
    
    # --- Model 2: Hawkes ---
    try:
        hawkes = HawkesExp(baseline_model=baseline_model).fit(train)
        
        # Metrics
        rmse_h, _, _ = evaluate_forecast(hawkes, train, test)
        ll_h = calc_test_ll(hawkes, train, test)
        
        # Для чистоты эксперимента, "Пуассон" - это Hawkes с alpha=0, beta=1 (не важно)
        # Или можно создать отдельный инстанс и не фитить его параметры, а занулить
        # Но у нас есть метрики! Давай просто посчитаем метрики для baseline_model
        # Нам придется чуть схитрить, так как evaluate_forecast ждет объект с get_intensity
        
        # Считаем RMSE для чистого Пуассона (Baseline)
        rmse_p, _, _ = evaluate_forecast(baseline_model, train, test)
        
        # LL для Пуассона (нужен метод nll, которого нет у UserBaseline)
        # Можно добавить nll в UserBaseline или посчитать Hawkes с params=[0, 1] (alpha=0)
        # Давай второй вариант, он проще
        poisson_proxy = HawkesExp(baseline_model=baseline_model)
        poisson_proxy.params = [0.0, 1.0] # Alpha=0 -> чисто сезонность
        ll_p = calc_test_ll(poisson_proxy, train, test)

        results.append({
            'user_id': uid,
            'events_count': len(times),
            'rmse_hawkes': rmse_h,
            'rmse_poisson': rmse_p,
            'll_hawkes': ll_h,
            'll_poisson': ll_p,
            'improvement_rmse': (1 - rmse_h/rmse_p) * 100
        })
        
    except Exception as e:
        print(f"Error on user {uid}: {e}")

# 5. Анализ результатов
df_res = pd.DataFrame(results)
print(f"\nAverage RMSE Improvement: {df_res['improvement_rmse'].mean():.2f}%")
print(f"Hawkes Win Rate (LL): {(df_res['ll_hawkes'] > df_res['ll_poisson']).mean()*100:.1f}%")
print(df_res.describe())